In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "default"

bronze_trips = spark.table(f"{CATALOG}.{SCHEMA}.trips_raw")
silver_trips = spark.table(f"{CATALOG}.{SCHEMA}.trips_clean")
fact_trips = spark.table(f"{CATALOG}.{SCHEMA}.fact_trips")
dim_location = spark.table(f"{CATALOG}.{SCHEMA}.dim_location")

checks_passed, checks_failed = [], []

def check(name, condition):
    (checks_passed if condition else checks_failed).append(name)
    status = "PASS" if condition else "FAIL"
    print(f"[{status}] {name}")

bronze_count = bronze_trips.count()
silver_count = silver_trips.count()
gold_count = fact_trips.count()

check("silver row count <= bronze row count", silver_count <= bronze_count)
check("gold fact row count == silver row count", gold_count == silver_count)

print(f"bronze={bronze_count:,}  silver={silver_count:,}  gold={gold_count:,}")

In [0]:
# Nulls in required fact columns
required_cols = [
    "trip_id", "pickup_date_key", "pickup_location_key",
    "dropoff_location_key", "fare_amount", "total_amount",
]

null_counts = fact_trips.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in required_cols
]).collect()[0].asDict()

for col, n in null_counts.items():
    check(f"no nulls in fact_trips.{col}", n == 0)

# Referential integrity
valid_locations = set(row.location_key for row in dim_location.select("location_key").collect())

orphan_pickups = fact_trips.filter(~F.col("pickup_location_key").isin(valid_locations)).count()
orphan_dropoffs = fact_trips.filter(~F.col("dropoff_location_key").isin(valid_locations)).count()

check("no orphan pickup_location_key values", orphan_pickups == 0)
check("no orphan dropoff_location_key values", orphan_dropoffs == 0)

# Value range sanity checks
bad_fares = fact_trips.filter(F.col("fare_amount") <= 0).count()
bad_durations = fact_trips.filter(
       (F.col("trip_duration_minutes") < 0) | (F.col("trip_duration_minutes") > 240)
   ).count()
bad_passengers = fact_trips.filter(~F.col("passenger_count").between(1, 6)).count()

check("no non-positive fares", bad_fares == 0)
check("no trip durations outside 0-240 min", bad_durations == 0)
check("no passenger counts outside 1-6", bad_passengers == 0)



In [0]:
# Check: trip_id is unique (primary key constraint)
duplicate_trip_ids = (
    fact_trips.groupBy("trip_id").count().filter(F.col("count") > 1).count()
)
check("trip_id is unique", duplicate_trip_ids == 0)

# Check: payment_type_key only contains accepted values (1-6, per TLC data dictionary)
accepted_payment_types = {1, 2, 3, 4, 5, 6}
bad_payment_types = fact_trips.filter(
    ~F.col("payment_type_key").isin(accepted_payment_types)
).count()
check("payment_type_key only contains accepted values (1-6)", bad_payment_types == 0)

print(f"\n{len(checks_passed)} passed, {len(checks_failed)} failed")
if checks_failed:
    raise AssertionError(f"Data quality checks failed: {checks_failed}")
print("All data quality checks passed.")

In [0]:

# Summary
print(f"\n{len(checks_passed)} passed, {len(checks_failed)} failed")
if checks_failed:
    raise AssertionError(f"Data quality checks failed: {checks_failed}")
print("All data quality checks passed.")